# A-3 · New customers and retention by age band (Stream A should-have)
**Stream A, Person A** · supports slide 4 · follows up on A-1 finding F3/F4

A-1 showed 18–24 revenue fell ~45% and never recovered, driven by fewer active customers, not smaller baskets.
This notebook asks **why there are fewer active young customers**:
1. **Acquisition:** did fewer *new* 18–24 customers start shopping after the competitor opened?
2. **Retention:** did *existing* 18–24 customers stop coming back, and were they less loyal than other ages even **before** the competitor?
3. Is the 2026 dip (A-1 recovery index 53) still young customers leaving, or something else?

**Method**
- **New customer** = `signup_date` (the date of the first purchase). We only count sign-ups from **Nov 2022 onward**. The customer file only contains people who bought something in the sales window, so earlier sign-up years are missing everyone who stopped shopping before Nov 2022 (survivorship bias).
- **Retention** = share of customers who bought in a window and **bought again in the same window one year later**. Using the same months each year cancels out seasonality.
  - **Nov–Feb windows** give us a **pre-competitor benchmark**: Nov 2022–Feb 2023 → Nov 2023–Feb 2024 sits entirely before the competitor opened (Mar 2024).
  - Mar–Oct windows are shown too, to match A-1.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.metrics import load
from src import config as C

OUT = C.RESULTS_DIR / "a_competitor"
OUT.mkdir(parents=True, exist_ok=True)
AGE_ORDER = ["18-24", "25-34", "35-44", "45-54", "55+"]
AGE_COLORS = {"18-24": "#D1495B", "25-34": "#E09F3E", "35-44": "#2B4C7E", "45-54": "#5B8DB8", "55+": "#9AA5B1"}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold",
                     "axes.titlelocation": "left", "axes.grid": True, "grid.alpha": 0.25,
                     "legend.frameon": False})

lines = load("lines")
customers = load("customers")
age = customers.set_index("customer_id").age_band
p = lines[lines.row_type == C.PURCHASE]

## 1. New customers per quarter, by age band

In [ ]:
new = customers[customers.signup_date >= C.DATA_START].copy()
new["quarter"] = new.signup_date.dt.to_period("Q")
q = new.pivot_table(index="quarter", columns="age_band", values="customer_id", aggfunc="count").fillna(0)
q = q.loc[: (C.DATA_END - pd.DateOffset(months=1)).to_period("Q") - 1, AGE_ORDER]   # drop the partial last quarter (Oct 2026 only)

fig, ax = plt.subplots(figsize=(10, 4.3))
x = q.index.to_timestamp()
for a in AGE_ORDER:
    hit = a == "18-24"
    ax.plot(x, q[a], marker="o", ms=3.5, color=AGE_COLORS[a], lw=2.6 if hit else 1.3, label=a, zorder=3 if hit else 2)
ax.axvline(C.COMPETITOR_OPEN, color="#555", ls="--", lw=0.9)
ax.text(C.COMPETITOR_OPEN, ax.get_ylim()[1] * 0.97, " Competitor opens\n (Mar 2024)", fontsize=8.5, va="top")
ax.set(title="New 18–24 customers roughly halved after the competitor opened, and never recovered",
       ylabel="New customers per quarter")
ax.legend(ncol=5, loc="lower center", bbox_to_anchor=(0.5, -0.22), fontsize=9)
plt.tight_layout(); fig.savefig(OUT / "a3_new_customers_by_age.png", bbox_inches="tight"); plt.show()

In [ ]:
# Same comparison as A-1: Mar–Oct each year
mo = new[new.signup_date.dt.month.between(3, 10)]
by_year = mo.pivot_table(index=mo.signup_date.dt.year, columns="age_band", values="customer_id", aggfunc="count")[AGE_ORDER]
by_year.index.name = "Mar–Oct of"
share = by_year.div(by_year.sum(axis=1), axis=0)
vs23 = by_year / by_year.loc[2023] - 1

display(pd.concat({"New customers": by_year, "Share of new customers": share, "vs. 2023": vs23}, axis=1)
        .style.format("{:,.0f}", subset="New customers")
        .format("{:.0%}", subset="Share of new customers")
        .format("{:+.0%}", subset="vs. 2023")
        .set_caption("<b>New customers in Mar–Oct, by age band</b>")
        .set_table_styles([{"selector": "caption", "props": "caption-side: top; text-align: left;"}]))

ch = mo[mo.age_band == "18-24"].pivot_table(index=mo.signup_date.dt.year, columns="acquisition_channel",
                                             values="customer_id", aggfunc="count").fillna(0).astype(int)
ch.index.name = "Mar–Oct of"
print("New 18-24 customers by acquisition channel:")
ch

## 2. Do existing customers come back? Retention by age band
Share of customers who bought in a window and bought again in the same window a year later.

In [ ]:
def active(start, end):
    return set(p[(p.date >= start) & (p.date < end)].customer_id)

def retention(w_from, w_to):
    base = pd.Series(sorted(active(*w_from)))
    back = base.isin(active(*w_to))
    r = back.groupby(base.map(age).values).mean().reindex(AGE_ORDER)
    r["25+ (all other ages)"] = back[base.map(age).isin(AGE_ORDER[1:]).values].mean()
    return r

NOV_FEB = {2022: ("2022-11-01", "2023-03-01"), 2023: ("2023-11-01", "2024-03-01"),
           2024: ("2024-11-01", "2025-03-01"), 2025: ("2025-11-01", "2026-03-01")}
ret_nf = pd.DataFrame({
    "Nov 22–Feb 23 → Nov 23–Feb 24\n(before competitor)": retention(NOV_FEB[2022], NOV_FEB[2023]),
    "Nov 23–Feb 24 → Nov 24–Feb 25\n(competitor opens in between)": retention(NOV_FEB[2023], NOV_FEB[2024]),
    "Nov 24–Feb 25 → Nov 25–Feb 26\n(after)": retention(NOV_FEB[2024], NOV_FEB[2025]),
})
MAR_OCT = {y: (f"{y}-03-01", f"{y}-11-01") for y in (2023, 2024, 2025, 2026)}
ret_mo = pd.DataFrame({f"Mar–Oct {y} → {y + 1}": retention(MAR_OCT[y], MAR_OCT[y + 1]) for y in (2023, 2024, 2025)})

display(ret_nf.style.format("{:.0%}").set_caption("<b>Retention, Nov–Feb windows</b> (includes a pre-competitor benchmark)")
        .set_table_styles([{"selector": "caption", "props": "caption-side: top; text-align: left;"}]))
display(ret_mo.style.format("{:.0%}").set_caption("<b>Retention, Mar–Oct windows</b> (same window as A-1)")
        .set_table_styles([{"selector": "caption", "props": "caption-side: top; text-align: left;"}]))
gap = ret_nf.loc["18-24"] - ret_nf.loc["25+ (all other ages)"]
print("18-24 minus 25+ retention gap (Nov–Feb):", ", ".join(f"{g:+.0%}" for g in gap))

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4))
labels = ["Before competitor\n(Nov 22–Feb 23 → next winter)", "Competitor opens in between\n(Nov 23–Feb 24 → next winter)",
          "After\n(Nov 24–Feb 25 → next winter)"]
xx = np.arange(3); w = 0.36
for i, (row, col, name) in enumerate([("18-24", "#D1495B", "18–24"), ("25+ (all other ages)", "#2B4C7E", "25 and over")]):
    vals = ret_nf.loc[row].values
    ax.bar(xx + (i - 0.5) * w, vals, w, color=col, label=name)
    for xi, v in zip(xx + (i - 0.5) * w, vals):
        ax.text(xi, v + 0.01, f"{v:.0%}", ha="center", fontsize=9)
ax.set_xticks(xx, labels, fontsize=8.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set(ylim=(0, 0.68), ylabel="Bought again next winter",
       title="Young customers were as loyal as everyone before the competitor; afterwards they were the least likely to return")
ax.legend(loc="upper right")
plt.tight_layout(); fig.savefig(OUT / "a3_retention_by_age.png", bbox_inches="tight"); plt.show()

## 3. Where did the active 18–24 customers go? New vs. returning
Active 18–24 customers in Mar–Oct = **new** that year (signed up in that window) + **returning** (signed up earlier).

In [ ]:
rows = {}
for y in (2023, 2024, 2025, 2026):
    ids = pd.Series(sorted(active(*MAR_OCT[y])))
    ids = ids[ids.map(age).eq("18-24").values]
    sd = customers.set_index("customer_id").signup_date.reindex(ids)
    is_new = (sd >= MAR_OCT[y][0]).values
    rows[y] = {"new": int(is_new.sum()), "returning": int((~is_new).sum())}
split = pd.DataFrame(rows).T
split["total active"] = split.sum(axis=1)
display(split)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(split.index.astype(str), split["returning"], color="#5B8DB8", label="Returning (signed up earlier)")
ax.bar(split.index.astype(str), split["new"], bottom=split["returning"], color="#D1495B", label="New that year")
for i, (y, r) in enumerate(split.iterrows()):
    ax.text(i, r["total active"] + 15, f"{r['total active']:,}", ha="center", fontsize=9, fontweight="bold")
    ax.text(i, r["returning"] + r["new"] / 2, f"{r['new']}", ha="center", va="center", color="white", fontsize=8.5)
    ax.text(i, r["returning"] / 2, f"{r['returning']}", ha="center", va="center", color="white", fontsize=8.5)
ax.set(title="Active 18–24 customers (Mar–Oct) shrank as fewer new ones replaced those who left",
       ylabel="Active 18–24 customers", ylim=(0, split["total active"].max() * 1.15))
ax.legend(loc="upper right", fontsize=8.5)
plt.tight_layout(); fig.savefig(OUT / "a3_18_24_new_vs_returning.png", bbox_inches="tight"); plt.show()

d = split.loc[2024] - split.loc[2023]
print(f"2023 -> 2024: active 18-24 {d['total active']:+,}  (new {d['new']:+,}, returning {d['returning']:+,})")

## 4. Headline numbers to hand over

In [ ]:
summary = {
    "New 18-24 customers, Mar–Oct 2023 → 2024 / 2025 / 2026":
        f"{by_year.loc[2023, '18-24']} → {by_year.loc[2024, '18-24']} / {by_year.loc[2025, '18-24']} / {by_year.loc[2026, '18-24']} "
        f"({vs23.loc[2024, '18-24']:+.0%} / {vs23.loc[2025, '18-24']:+.0%} / {vs23.loc[2026, '18-24']:+.0%})",
    "18-24 share of new customers, 2023 → 2026": f"{share.loc[2023, '18-24']:.0%} → {share.loc[2026, '18-24']:.0%}",
    "New 25+ customers, Mar–Oct 2024 vs 2023": f"{by_year.loc[2024, AGE_ORDER[1:]].sum() / by_year.loc[2023, AGE_ORDER[1:]].sum() - 1:+.0%}",
    "Retention 18-24 vs 25+, before competitor": f"{ret_nf.iloc[:, 0]['18-24']:.0%} vs {ret_nf.iloc[:, 0]['25+ (all other ages)']:.0%}",
    "Retention 18-24 vs 25+, across competitor opening": f"{ret_nf.iloc[:, 1]['18-24']:.0%} vs {ret_nf.iloc[:, 1]['25+ (all other ages)']:.0%}",
    "Retention 18-24 vs 25+, after": f"{ret_nf.iloc[:, 2]['18-24']:.0%} vs {ret_nf.iloc[:, 2]['25+ (all other ages)']:.0%}",
    "Active 18-24 change 2023→2024 (new / returning)": f"{d['new']:+,} / {d['returning']:+,}",
    "Active 18-24, Mar–Oct 2023 → 2026": f"{split.loc[2023, 'total active']:,} → {split.loc[2026, 'total active']:,}",
}
pd.Series(summary, name="value").to_frame()